# 🔥 Predicción de Incendios Forestales en EE.UU. (2014-2025)

Dataset: [US Wildfire Dataset (2014-2025)](https://www.kaggle.com/datasets/firecastrl/us-wildfire-dataset)

## Descripción del dataset

El dataset **US Wildfire** fue obtenido de Kaggle y contiene registros climáticos y geográficos de puntos relevados a lo largo de los Estados Unidos, con el objetivo de predecir si en una ubicación y momento dados ocurrió o no un incendio forestal.

- **Variable objetivo:** `Wildfire` (indica si hubo incendio o no)

## Justificación de la elección

Los incendios forestales representan una de las catástrofes naturales más devastadoras del planeta, con un impacto directo sobre ecosistemas, comunidades y el cambio climático. EE.UU. es uno de los países más afectados, especialmente en el oeste del país.

Este dataset permite abordar un problema de **clasificación binaria** real y relevante: predecir si las condiciones climáticas y ambientales de un lugar determinado son propicias para que ocurra un incendio. Las variables disponibles cubren factores clave como temperatura, humedad, viento y radiación solar, lo que lo convierte en un dataset ideal para aplicar técnicas de Machine Learning supervisado.

In [ ]:
# ==========================================================
# IMPORTO LAS LIBRERÍAS PRINCIPALES
# ==========================================================

# importamos librerías para descargar datasets desde kaggle
from pathlib import Path
import kagglehub

# Importamos librerías generales para trabajar con datos
import os
import pandas as pd

# Importamos matplotlib y seaborn para realizar gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas para separar datos y validar modelos
from sklearn.model_selection import train_test_split, cross_val_score

# Herramientas para construir pipelines
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Herramientas de preprocesamiento
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Modelos de clasificación
from sklearn.linear_model import LogisticRegression

# Métricas de evaluación para clasificación
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score
)

#import joblib

# Configuración de estilo para los gráficos
sns.set(style='whitegrid')
# Configuración general para que pandas muestre mejor las tablas
pd.set_option("display.max_columns", None)

In [ ]:
# ==========================================================
# CARGO EL DATASET
# ==========================================================

PROJECT_ROOT = Path.cwd()
DATASET_PATH = PROJECT_ROOT / "data" / "Wildfire_dataset.csv"

# Si el dataset no existe localmente, lo descargamos desde Kaggle
if not DATASET_PATH.exists():
    print("No se encontró el dataset localmente.")
    print("Descargando el dataset desde Kaggle...")

    downloaded_path = kagglehub.dataset_download(
        "firecastrl/us-wildfire-dataset",
        path="Wildfire_Dataset.csv",
        output_dir=str(PROJECT_ROOT / "data"),
    )

    DATASET_PATH = Path(downloaded_path)
    print(f"Dataset descargado en: {DATASET_PATH}")
else:
    print(f"Dataset encontrado en: {DATASET_PATH}")
df = pd.read_csv(DATASET_PATH)

In [ ]:
# ==========================================================
# ANALIZO EL DATASET
# ==========================================================

print("Dimensiones del dataset:", df.shape)
df.info(show_counts=True)

## Variables del dataset

| Variable | Tipo | Descripción |
|----------|------|-------------|
| `latitude` | float64 | Latitud geográfica del punto |
| `longitude` | float64 | Longitud geográfica del punto |
| `datetime` | object | Fecha y hora del registro |
| `Wildfire` | object | Variable objetivo: incendio o no |
| `pr` | float64 | Precipitación |
| `rmax` | float64 | Humedad relativa máxima |
| `rmin` | float64 | Humedad relativa mínima |
| `sph` | float64 | Humedad específica |
| `srad` | float64 | Radiación solar |
| `tmmn` | float64 | Temperatura mínima |
| `tmmx` | float64 | Temperatura máxima |
| `vs` | float64 | Velocidad del viento |
| `bi` | float64 | Índice de quema (Burning Index) |
| `fm100` | float64 | Humedad del combustible muerto a 100 horas |
| `fm1000` | float64 | Humedad del combustible muerto a 1000 horas |
| `erc` | float64 | Componente de liberación de energía |
| `etr` | float64 | Evapotranspiración de referencia |
| `pet` | float64 | Evapotranspiración potencial |
| `vpd` | float64 | Déficit de presión de vapor |

In [ ]:
# --------------------------------------------------------------
# Primer vistazo del dataset
# --------------------------------------------------------------
print('\nPrimeras 5 filas del dataset:')
display(df.head())

In [ ]:
print('\nResumen estadístico de las variables numéricas:')
display(df.describe())

##### Acá hay algo extraño y es que el max de muchas columnas es de 32767 exactos (Esto es revisado más adelante).

In [ ]:
print('\nValores faltantes por columna:')
display(df.isnull().sum())

##### El dataset está muy limpio. No tiene ni un solo valor nulo.

## Separación de variables numéricas y categóricas

In [ ]:
print('\nColumnas numéricas:')
display(df.select_dtypes(include='number').columns)

print('\nColumnas no numéricas:')
display(df.select_dtypes(exclude='number').columns)

##### Son todas variables numéricas sin contar la variable objetivo (Wildfire) y las fechas. Como no hay variables categóricas, no voy a tener que hacer ninguna transformación usando Label Encoding o One-Hot Encoding

## Distribución de la variable objetivo

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Wildfire')
plt.title('Distribución de la variable Wildfire')
plt.xlabel('Wildfire')
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
# El dataset está desbalanceado porque hay solo un 5% de incendios
df['Wildfire'].value_counts(normalize=True) * 100

##### La variable objetivo está muy desbalanceada

In [ ]:
# Hago una copia para trabajar
df_transformado = df.copy()

# Convierto la columna datetime a formato fecha
df_transformado['datetime'] = pd.to_datetime(
    df_transformado['datetime'], format='%Y-%m-%d'
)

# Convierto la columna Wildfire a formato bool
df_transformado['Wildfire'] = df_transformado['Wildfire'].map({'Yes': 1, 'No': 0})

# Verifico que todo esté bien
print(df_transformado.dtypes)
print("-"*40)
print(df_transformado[df_transformado['Wildfire'] == 1][['datetime', 'Wildfire']].sample(5))
print(df_transformado[df_transformado['Wildfire'] == 0][['datetime', 'Wildfire']].sample(5))

In [ ]:
# Reviso los duplicados
print('Duplicados totales: ', df_transformado.duplicated().sum())

# Los elimino
df_transformado = df_transformado.drop_duplicates()
print('Duplicados eliminados.')

# Reviso si fueron eliminados
print('Duplicados restantes: ', df_transformado.duplicated().sum())

### Reviso el valor extraño que se repetía tanto en el describe

In [ ]:
# Primero voy a crear una máscara para que me muestre solo las filas con ese valor
cols_numericas = df_transformado.select_dtypes(include='number').columns
mascara = (df_transformado[cols_numericas] == 32767).any(axis=1)
print('Filas con valor 32767:', mascara.sum())

In [ ]:
# Reviso qué porcentaje de estos valores representan un incendio positivo.
print(df_transformado[mascara]['Wildfire'].value_counts())
print(df_transformado[mascara]['Wildfire'].value_counts(normalize=True) * 100)

##### Los valores representan un 100% de valores negativos, por lo que no tienen ninguna influencia en los incendios.

In [ ]:
# Doy un vistazo a las primeras 10 filas y veo que todos los valores salvo fecha, son iguales
df_transformado[mascara].head(10)

In [ ]:
# Los elimino, quedandome con todos los datos opuestos a la mascara que creé
df_transformado = df_transformado[~mascara]

#### **Conclusión:** Me parece que la mejor idea fue eliminarlos, ya que al ser todos datos iguales en la misma ubicación geográfica, estimo que son filas donde el sensor no registró ningún dato válido y se tuvo que rellenar con ese valor que no significa nada, pero puede llegar a afectar al modelo.

## Outliers

In [ ]:
# Reviso qué cantidad de datos extremos hay en cada columna
for col in cols_numericas:
    Q1 = df_transformado[col].quantile(0.25)
    Q3 = df_transformado[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df_transformado[col] < Q1 - 1.5 * IQR) | (df_transformado[col] > Q3 + 1.5 * IQR)).sum()
    print(f'{col}: {outliers} outliers ({outliers/len(df_transformado)*100:.2f}%)')

In [ ]:
# Muestro el porcentaje de True Wildfires que hay en esos valores extremos
for col in cols_numericas:
    Q1 = df_transformado[col].quantile(0.25)
    Q3 = df_transformado[col].quantile(0.75)
    IQR = Q3 - Q1
    mascara_outlier = (df_transformado[col] < Q1 - 1.5 * IQR) | (df_transformado[col] > Q3 + 1.5 * IQR)
    total = mascara_outlier.sum()
    if total > 0:
        pct_wildfire = df_transformado[mascara_outlier]['Wildfire'].mean() * 100
        print(f'{col}: {total} outliers - {pct_wildfire:.2f}% son incendios')

#### **Conclusión:** Mi desición final fue **no eliminarlos**. pr (precipitación) es el más llamativo ya que hay un 24.35% de valores extremos, pero que hayan muchos días sin lluvia y pocos con mucha lluvia es algo normal, por lo que no creo que sean outliers. El resto están todos por debajo del 4% y son variables climáticas donde los extremos tienen sentido. Así que dicidí ver qué porcentaje de True Wildfires hay en esos valores extremos. Estos outliers no son errores, sino fenómenos climáticos reales con cierta correlación con los incendios, por lo tanto los conservo.

In [ ]:
# Creo las columnas año, mes y día a partir de la columna datetime y luego la elimino.
from datetime import date

df_transformado['anio'] = df_transformado['datetime'].dt.year
df_transformado['mes'] = df_transformado['datetime'].dt.month
df_transformado['dia'] = df_transformado['datetime'].dt.day
df_transformado = df_transformado.drop(columns=['datetime'])

In [ ]:
display(df_transformado.head())

In [ ]:
# Muestro las correlaciones entre las columnas y la variable objetivo para saber cuales usar como variables predictoras
correlaciones = df_transformado.corr()['Wildfire'].sort_values(ascending=False)
print(correlaciones)

##### La correlación con las columnas es muy mala, por lo que ya me estoy esperando que el modelo no va a ser muy bueno

In [ ]:
# Voy a sacar año, a pesar de que tiene la correlación más alta, ya que no es parámetro. Porque hay años donde hubo más incendios que otros y eso puede introducir un sesgo temporal
variables_predictoras = ['mes', 'erc', 'bi', 'vpd', 'tmmx', 'tmmn', 'etr', 'pet',
            'fm1000', 'fm100', 'rmin', 'rmax', 'latitude', 'longitude']

variable_objetivo = 'Wildfire'

X = df_transformado[variables_predictoras]
y = df_transformado[variable_objetivo]

print('Variables predictoras seleccionadas:')
print(variables_predictoras)

print('\nVariable objetivo:')
print(variable_objetivo)

print('\nDimensiones de X:', X.shape)
print('Dimensiones de y:', y.shape)

display(X.head())
display(y.head())

## División estratificada de los datos

La división entre entrenaimento y prueba es practicamente la misma proporción que tenía el dataset original gracias a stratify=y

In [ ]:
# Divido los datos en entrenamiento y prueba
# Uso el 80% de los datos para entrenar el modelo y el 20% para evaluar el rendimiento

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print('Dimensiones de X_train:', X_train.shape)
print('Dimensiones de X_test:', X_test.shape)
print('Dimensiones de y_train:', y_train.shape)
print('Dimensiones de y_test:', y_test.shape)

print('\nDistribuciones de clases en y_train:')
display(y_train.value_counts(normalize=True) * 100)

print('\nDistribución de clases en y_test:')
display(y_test.value_counts(normalize=True) * 100)

## Construcción del preprocesamiento

In [ ]:
# Identificamos las columnas numéricas

columnas_numericas = X_train.columns

# Transformador para variables numéricas
transformador_numerico = Pipeline(
    steps=[
        ("imputador", SimpleImputer(strategy="median")), # Por si hay valores faltantes
        ("escalador", StandardScaler())
    ]
)

# Mostramos la cantidad de columnas numéricas
print("Cantidad de columnas numéricas:", len(columnas_numericas))

In [ ]:
# ---------------------------------------------------------
# Construcción del preprocesamiento
# ---------------------------------------------------------

# Transformador general:
# - escala columnas numéricas
preprocesamiento = ColumnTransformer(
    transformers=[
        ("numericas", transformador_numerico, columnas_numericas)
    ]
)

print("Preprocesador creado:")
print(preprocesamiento)

## Creación de pipeline

In [ ]:
# ---------------------------------------------------------
# Creación del pipeline completo
# ---------------------------------------------------------

# Creamos el modelo que usaremos al final del pipeline
# Acá uso 1000 iteraciones
modelo_logistico = LogisticRegression(
    max_iter=1000,
    class_weight='balanced', # Pongo balanced ya que la variable objetivo tiene una distribución desbalanceada
    random_state=42
)

# Creamos el pipeline completo
pipeline_lr = Pipeline(
    steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", modelo_logistico)
    ]
)

# Entrenamos el pipeline completo
pipeline_lr.fit(X_train, y_train)

print("Modelo de Regresión Logística entrenado correctamente.")

In [ ]:
# Realizo predicciones con el modelo

y_pred = pipeline_lr.predict(X_test)
y_proba = pipeline_lr.predict_proba(X_test)

print('Primeras 25 clases reales:')
print(y_test.values[:25])

print('\nPrimeras 25 clases predichas:')
print(y_pred[:25])

print('\nPrimeras 25 probabilidades estimadas:')
display(pd.DataFrame(
    y_proba[:25],
    columns=['Probabilidad incendio positivo', 'Probabilidad incendio negativo']
))

##### Acá ya estoy notando que el modelo está prediciendo muchos falsos positivos

In [ ]:
# Calculo el accuracy del modelo

accuracy = accuracy_score(y_test, y_pred)

print('Accuracy del modelo:', accuracy)
print(f'Accuracy del modelo: {accuracy:.2%}')

#### Pésimo accuracy. Sospecho que es porque todas las variables predictivas tenían poca correlación con la variable objetivo :(

In [ ]:
# Calculo precisión, reacall y F1-SCORE

reporte = classification_report(
    y_test,
    y_pred,
    target_names=["Incendio negativo", "Incendio positivo"]
)

print(reporte)

- **Accuracy (58%):** De todos los casos, ¿cuántos calificó bien?. Es la métrica más engañosa, ya que es un dataset que tiene un 95% de negativos. Un modelo que predice siempre "No incendio" puede tener un accuracy muy alto y aun así no aprender nada útil.

- **Precisión (7%):** De todos los que predijo como positivo, ¿cuántos realmente lo eran?. 7% es muy bajo, significa que el modelo detecta muchísimos falsos positivos.

- **Recall (59%):** De todos los positivos, ¿Cuántos realmente lo eran?. En este caso, la métrica más importante ya que el costo de no detectar un positivo real es el más alto. 59% es la métrica más alta, pero sigue siendo un número bajo.

- **F1-score (13%):** Promedio entre precisión y recall. 13% es muy bajo.

**Conclusión:** Es un modelo muy débil. No es bueno prediciendo.

In [ ]:
# Validación cruzada

scores_recall = cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring='recall')
scores_precision = cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring='precision')
scores_f1 = cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring='f1')

print(f"Recall por fold: {scores_recall}")
print(f"Recall promedio: {scores_recall.mean():.3f} (+/- {scores_recall.std():.3f})")

print(f"Precision por fold: {scores_precision}")
print(f"Precision promedio: {scores_precision.mean():.3f} (+/- {scores_precision.std():.3f})")

print(f"F1 por fold: {scores_f1}")
print(f"F1 promedio: {scores_f1.mean():.3f} (+/- {scores_f1.std():.3f})")

La validación cruzada (5 folds) confirma que las métricas obtenidas en el test set no fueron producto del azar:

- **Recall: 0.594 (± 0.002):** El modelo detecta ~59% de los incendios reales de forma consistente entre folds. La desviación mínima indica que este comportamiento es estable, no depende del split de datos.
- **Precision: 0.072 (± 0.000):** De cada 100 predicciones de "Wildfire", solo ~7 son correctas. Es decir, el modelo genera muchas falsas alarmas.
- **F1-score: 0.129 (± 0.000):** Bajo, como consecuencia directa del desbalance entre recall y precision.

In [ ]:
# Calculo y grafico la matriz de confusión

matriz = confusion_matrix(y_test, y_pred)

print("Matriz de confusión:")
print(matriz)

# Muestro la matriz como gráfico

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["Incendio negativo", "Incedio positivo"]
)

disp.plot(values_format="d")
plt.title("Matriz de confusión - Regresión logística")
plt.show()

In [ ]:
# ==========================================================
# CALCULAMOS MÉTRICAS A PARTIR DE LA MATRIZ DE CONFUSIÓN
# ==========================================================
# Extraemos los valores de la matriz.
#
# La matriz tiene esta estructura:
#
# [[VN, FP],
# [FN, VP]]
VN, FP, FN, VP = matriz.ravel()
print("Verdaderos negativos (VN):", VN)
print("Falsos positivos (FP):", FP)
print("Falsos negativos (FN):", FN)
print("Verdaderos positivos (VP):", VP)
# ----------------------------------------------------------
# Calculamos precision y recall para la clase positiva
# ----------------------------------------------------------
precision_positiva = VP / (VP + FP)
recall_positivo = VP / (VP + FN)
f1_positivo = 2 * (precision_positiva * recall_positivo) / (precision_positiva + recall_positivo)

print("\nMétricas para la clase positiva:")
print(f"Precision: {precision_positiva:.2f}")
print(f"Recall: {recall_positivo:.2f}")
print(f"F1-score: {f1_positivo:.2f}")

|  | Predicho: No incendio | Predicho: Incendio |
|---|---|---|
| **Real: No incendio** | 1.547.529 ✅ (Verdadero Negativo) | 1.143.245 ❌ (Falso Positivo) |
| **Real: Incendio** | 61.152 ❌ (Falso Negativo) | 89.158 ✅ (Verdadero Positivo) |

**Interpretación:**
- El modelo detectó correctamente **89.158 incendios reales** (Verdaderos Positivos)
- Sin embargo, generó **1.143.245 falsas alarmas** (Falsos Positivos) — casos clasificados como incendio que no lo eran
- Dejó sin detectar **61.152 incendios reales** (Falsos Negativos)

> Dado que el costo de no detectar un incendio real es muy alto, el **recall** es la métrica más relevante para este problema. El modelo alcanza un recall de **59%**, lo que significa que detecta poco más de la mitad de los incendios reales.

In [ ]:
# ==========================================================
# PROBAMOS DISTINTOS UMBRALES DE DECISIÓN
# ==========================================================

# Tomamos solamente la probabilidad de la clase positiva.
# Es decir, la probabilidad de que sí haya incendio.
probabilidad_clase_positiva = y_proba[:, 1]
umbrales = [0.3, 0.4, 0.5, 0.6, 0.7]
resultados_umbrales = []
for umbral in umbrales:
    # Si la probabilidad de clase positiva supera el umbral,
    # clasificamos el caso como 1. Si no, como 0.
    y_pred_umbral = (probabilidad_clase_positiva >= umbral).astype(int)

    precision = precision_score(y_test, y_pred_umbral)
    recall = recall_score(y_test, y_pred_umbral)
    f1 = f1_score(y_test, y_pred_umbral)

    resultados_umbrales.append({
        "umbral": umbral,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
})
resultados_umbrales = pd.DataFrame(resultados_umbrales)
display(resultados_umbrales)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(resultados_umbrales["umbral"], resultados_umbrales["precision"], marker="o", label="Precision")
plt.plot(resultados_umbrales["umbral"], resultados_umbrales["recall"], marker="o", label="Recall")
plt.plot(resultados_umbrales["umbral"], resultados_umbrales["f1_score"], marker="o", label="F1-score")

plt.title("Precision, recall y F1-score según el umbral de decisión")
plt.xlabel("Umbral de decisión")
plt.ylabel("Valor de la métrica")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.show()

El umbral de decisión define a partir de qué probabilidad el modelo clasifica un caso como incendio.
Por defecto es 0.5, pero puede ajustarse según lo que se quiera priorizar.

| Umbral | Precision | Recall | F1-score |
|--------|-----------|--------|----------|
| 0.3 | 5.5% | 98.5% | 10.4% |
| 0.4 | 6.2% | 88.5% | 11.5% |
| 0.5 | 7.2% | 59.3% | 12.9% |
| 0.6 | 8.6% | 23.9% | 12.6% |
| 0.7 | 10.1% | 2.5% | 4.0% |

**Interpretación:**
- A medida que se **sube el umbral**, la precision aumenta levemente pero el recall cae drásticamente
- El **mejor F1-score se obtiene en 0.5**, el umbral por defecto
- No existe un umbral que mejore significativamente ambas métricas a la vez, esto indica que la limitación es estructural: las variables predictoras no tienen suficiente poder predictivo.

Tal vez en un contexto real de detección de incendios, podría preferirse un umbral bajo (como 0.3) para maximizar el recall y detectar la mayor cantidad posible de incendios reales, aceptando más falsas alarmas

In [ ]:
# ==================
# CURVA ROC Y AUC
# ==================

y_proba = pipeline_lr.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Modelo aleatorio')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (Recall)')
plt.title('Curva ROC')
plt.legend()
plt.show()

print(f"AUC: {auc:.3f}")

## Interpretación de la Curva ROC

**AUC = 0.619**

El modelo tiene poder predictivo real, pero débil. Está por encima del azar (0.5), lejos de un modelo bueno (>0.8).

- La curva se separa de la diagonal en todo el rango, confirmando que el modelo distingue algo entre clases, no es puro ruido.
- 0.619 es consistente con lo ya observado: correlaciones bajas entre predictores y target.
- La curva se aplana en la zona de FPR alto (0.6-1.0): ganar recall adicional ahí cuesta cada vez más falsos positivos. Eso también contextualiza por qué la precision es tan baja en cualquier threshold razonable.

## **Conclusión del informe**
- El dataset no es del todo apto para el problema debido a las bajas correlaciones y un AUC de 0.619.
- Con 59% de recall, 4 de cada 10 incendios reales quedan sin detectar. No es aceptable para un sistema real.
- Con un 7% de precisión en un sistema real de alerta de incendios, se generarían alarmas falsas constantemente. Esto tiene un costo operativo enorme que tampoco es aceptable.
- El modelo captura señal real pero limitada. La causa no es el algoritmo (Logistic Regression) ni el threshold, es la capacidad predictiva de las variables disponibles. Mejorar esto requiere feature engineering adicional o un dataset con variables más correlacionadas al target, no ajustar hiperparámetros del modelo actual.